# Análise exploratória: COVID-19 e registros Brasil.IO

Marcelo Barbosa de Oliveira Junior. Execute `python -m src.etl --download` na raiz antes deste notebook. A base completa é agregada pelo ETL em blocos.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
p = root / "data" / "processed"
daily = pd.read_csv(p / "national_daily.csv.gz", parse_dates=["date"])
states = pd.read_csv(p / "state_latest.csv", parse_dates=["date"])
quality = pd.read_csv(p / "quality_by_state.csv")
manifest = pd.read_json(p / "manifest.json", typ="series")
manifest

## RQ1 — evolução temporal
A média móvel contém 7 datas consecutivas com as 27 UFs presentes. Casos e óbitos são comunicados na data do boletim.

In [ ]:
print(daily.loc[daily.cases_7d.idxmax(), ["date", "cases_7d"]])
print(daily.loc[daily.deaths_7d.idxmax(), ["date", "deaths_7d"]])
ax = daily.plot(x="date", y=["cases_7d", "deaths_7d"], subplots=True, figsize=(12, 6), title=["Casos/dia", "Óbitos/dia"], legend=False)
plt.tight_layout()

## RQ2 — diferenças entre UFs
Incidência acumulada por 100 mil habitantes no último registro disponível; confira a data de cada UF.

In [ ]:
display(states[["state", "date", "incidence_100k", "case_fatality_pct"]].sort_values("incidence_100k", ascending=False))
states.sort_values("incidence_100k").plot.barh(x="state", y="incidence_100k", figsize=(9, 8), legend=False)
plt.xlabel("Casos por 100 mil")

## RQ3 — repetição e revisões
Contagens usam somente linhas municipais; revisões negativas são mantidas.

In [ ]:
print(quality[["city_rows", "repeated", "negative_cases", "negative_deaths"]].sum())
print("Percentual repetido:", 100 * quality.repeated.sum()/quality.city_rows.sum())
quality.sort_values("repeated_pct", ascending=False).plot.bar(x="state", y="repeated_pct", figsize=(11, 4), legend=False)
plt.ylabel("% de linhas repetidas")